# 第 4 章 過学習・未学習と正則化

多項式回帰で次数を変えながら、訓練誤差とテスト誤差の差（汎化ギャップ）を観察し、L1 / L2 正則化の効果を確かめます。

対応する記事: [第 4 章 過学習・未学習と正則化（Kotlin Notebook の言語版）](../../../docs/article/grokking-machine-learning/kotlin/ch04.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch04.*

## データセット

`y = 2x + 3` に小さなノイズを乗せた 10 点です。**真の関係は 1 次関数** なので、高次のモデルは過学習するはずです。

In [2]:
val features = listOf(-1.5, -1.2, -0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9, 1.2)
val labels = listOf(0.08, 0.32, 1.07, 1.63, 2.54, 3.11, 3.84, 3.95, 4.75, 5.12)

val split = trainTestSplit(features, labels, testRatio = 0.3, seed = 0)
println("訓練 " + split.trainFeatures)
println("テスト " + split.testFeatures)

訓練 [0.0, -0.9, 0.9, -0.6, 0.3, -1.5, -0.3]
テスト [1.2, 0.6, -1.2]


## 次数を上げるとどうなるか

**訓練誤差は下がるのに、汎化ギャップ（テスト誤差 − 訓練誤差）は広がります。** これが過学習です。

In [3]:
println("%4s %10s %12s %10s".format("次数", "訓練 RMSE", "テスト RMSE", "ギャップ"))
listOf(1, 3, 5).forEach { degree ->
    val m = polynomialRegression(split.trainFeatures, split.trainLabels, degree = degree)
    val trainError = modelRmse(m, split.trainFeatures, split.trainLabels)
    val testError = modelRmse(m, split.testFeatures, split.testLabels)
    println("%4d %10.4f %12.4f %10.4f".format(degree, trainError, testError, testError - trainError))
}

  次数    訓練 RMSE     テスト RMSE       ギャップ


   1     0.1371       0.3324     0.1953


   3     0.0546       0.2922     0.2375
   5     0.0631       0.2963     0.2332


## 正則化を掛ける

5 次モデルに λ = 0.01 の正則化を掛けます。**訓練誤差は悪化しますが、テスト誤差は改善します。** 訓練データへの当てはまりをわざと諦めて、未知データへの当てはまりを買っています。

In [4]:
println("%-8s %8s %8s %10s".format("正則化", "訓練", "テスト", "重みの合計"))
listOf(Regularization.NONE, Regularization.L1, Regularization.L2).forEach { kind ->
    val strength = if (kind == Regularization.NONE) 0.0 else 0.01
    val m = polynomialRegression(split.trainFeatures, split.trainLabels, degree = 5,
                                 kind = kind, strength = strength)
    println("%-8s %8.4f %8.4f %10.3f".format(kind, modelRmse(m, split.trainFeatures, split.trainLabels),
            modelRmse(m, split.testFeatures, split.testLabels), weightMagnitude(m)))
}

正則化            訓練      テスト      重みの合計


NONE       0.0631   0.2963      2.753
L1         0.0814   0.2220      2.377


L2         0.1252   0.1807      2.843


## L1 は重みを 0 にする

学習された 5 つの重みを並べます。**L1 は不要な次数の重みをほぼ 0 まで押し下げます**（スパース性）。L2 は全体をなだらかに縮めるだけで、0 にはしません。

In [5]:
listOf(Regularization.NONE, Regularization.L1, Regularization.L2).forEach { kind ->
    val strength = if (kind == Regularization.NONE) 0.0 else 0.01
    val m = polynomialRegression(split.trainFeatures, split.trainLabels, degree = 5,
                                 kind = kind, strength = strength)
    println("%-6s %s".format(kind, m.weights.joinToString("  ") { "%8.4f".format(it) }))
}

NONE     2.1758   -0.2666    0.0350   -0.0944   -0.1810


L1       2.1034   -0.1990    0.0001    0.0002   -0.0740


L2       1.7452   -0.3407    0.4656    0.1211   -0.1706


## 試してみる

正則化の強さを変えるとどうなるでしょうか。**強すぎるとかえって悪化します。** 直感に反しますが、重みを潰しすぎると境界の位置そのものが崩れるためです。

In [6]:
listOf(0.0, 0.005, 0.01, 0.05, 0.2).forEach { strength ->
    val m = polynomialRegression(split.trainFeatures, split.trainLabels, degree = 5,
                                 kind = Regularization.L2, strength = strength)
    println("λ = %-6s テスト RMSE %.4f".format(strength, modelRmse(m, split.testFeatures, split.testLabels)))
}

λ = 0.0    テスト RMSE 0.2963
λ = 0.005  テスト RMSE 0.1618


λ = 0.01   テスト RMSE 0.1807
λ = 0.05   テスト RMSE 0.3536


λ = 0.2    テスト RMSE 0.6192
